[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/RAG_Techniques/blob/main/all_rag_techniques/simple_csv_rag_ollama.ipynb)

# Simple RAG (Retrieval-Augmented Generation) System for CSV Files using Ollama

## Overview

This code implements a basic Retrieval-Augmented Generation (RAG) system for processing and querying CSV documents. The system encodes the document content into a vector store, which can then be queried to retrieve relevant information.

# Ollama
Using Ollama, we can run this script without any API keys or credits on your personal computer using the small Llama3.2 model with just 2GB of memory. If you so wish to run this on colab, refer this [article](https://medium.com/@abonia/running-ollama-in-google-colab-free-tier-545609258453). To setup Ollama on MacOS, Windows or Linx refer the [official website](https://ollama.com/).

For more details on running Ollama, refer https://github.com/NirDiamant/agents-towards-production/blob/main/tutorials/on-prem-llm-ollama/ollama_tutorial.ipynb

# CSV File Structure and Use Case
The CSV file contains dummy customer data, comprising various attributes like first name, last name, company, etc. This dataset will be utilized for a RAG use case, facilitating the creation of a customer information Q&A system.

## Key Components

1. Loading and spliting csv files.
2. Vector store creation using [FAISS](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/) and OpenAI embeddings
3. Retriever setup for querying the processed documents
4. Creating a question and answer over the csv data.

## Method Details

### Document Preprocessing

1. The csv is loaded using langchain Csvloader
2. The data is split into chunks.


### Vector Store Creation

1. OpenAI embeddings are used to create vector representations of the text chunks.
2. A FAISS vector store is created from these embeddings for efficient similarity search.

### Retriever Setup

1. A retriever is configured to fetch the most relevant chunks for a given query.

## Benefits of this Approach

1. Scalability: Can handle large documents by processing them in chunks.
2. Flexibility: Easy to adjust parameters like chunk size and number of retrieved results.
3. Efficiency: Utilizes FAISS for fast similarity search in high-dimensional spaces.
4. Integration with Advanced NLP: Uses OpenAI embeddings for state-of-the-art text representation.

## Conclusion

This simple RAG system provides a solid foundation for building more complex information retrieval and question-answering systems. By encoding document content into a searchable vector store, it enables efficient retrieval of relevant information in response to queries. This approach is particularly useful for applications requiring quick access to specific information within a csv file.

import libries

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [1]:
# Install required packages
!pip install faiss-cpu langchain langchain-community pandas python-dotenv langchain-ollama

In [2]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="llama3.2")

# CSV File Structure and Use Case
The CSV file contains dummy customer data, comprising various attributes like first name, last name, company, etc. This dataset will be utilized for a RAG use case, facilitating the creation of a customer information Q&A system.

In [3]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/customers-100.csv https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv


--2025-06-01 15:50:36--  https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 206372 (202K) [application/octet-stream]
Saving to: ‘data/Understanding_Climate_Change.pdf’

data/Understanding_ 100%[===================>] 201.54K  --.-KB/s    in 0.05s   

2025-06-01 15:50:37 (4.34 MB/s) - ‘data/Understanding_Climate_Change.pdf’ saved [206372/206372]

--2025-06-01 15:50:37--  https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercon

In [4]:
import pandas as pd

file_path = ('data/customers-100.csv') # insert the path of the csv file
data = pd.read_csv(file_path)

#preview the csv file
data.head()

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,DD37Cf93aecA6Dc,Sheryl,Baxter,Rasmussen Group,East Leonard,Chile,229.077.5154,397.884.0519x718,zunigavanessa@smith.info,2020-08-24,http://www.stephenson.com/
1,2,1Ef7b82A4CAAD10,Preston,Lozano,Vega-Gentry,East Jimmychester,Djibouti,5153435776,686-620-1820x944,vmata@colon.com,2021-04-23,http://www.hobbs.com/
2,3,6F94879bDAfE5a6,Roy,Berry,Murillo-Perry,Isabelborough,Antigua and Barbuda,+1-539-402-0259,(496)978-3969x58947,beckycarr@hogan.com,2020-03-25,http://www.lawrence.com/
3,4,5Cef8BFA16c5e3c,Linda,Olsen,"Dominguez, Mcmillan and Donovan",Bensonview,Dominican Republic,001-808-617-6467x12895,+1-813-324-8756,stanleyblackwell@benson.org,2020-06-02,http://www.good-lyons.com/
4,5,053d585Ab6b3159,Joanna,Bender,"Martin, Lang and Andrade",West Priscilla,Slovakia (Slovak Republic),001-234-203-0635x76146,001-199-446-3860x3486,colinalvarado@miles.net,2021-04-17,https://goodwin-ingram.com/


load and process csv data

In [5]:
loader = CSVLoader(file_path=file_path)
docs = loader.load_and_split()

Initiate faiss vector store and openai embedding

In [6]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# Initialize Ollama embeddings
embeddings = OllamaEmbeddings(model="mxbai-embed-large")  # or another embedding model you have locally

# Create FAISS index using embedding dimension
index = faiss.IndexFlatL2(len(embeddings.embed_query(" ")))

# Create FAISS vector store
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

Add the splitted csv data to the vector store

In [7]:
vector_store.add_documents(documents=docs)

['c6cb3a4a-1af4-4c44-90b7-7a89d2b829f4',
 '7139309c-2347-41ce-9347-4d13565a71db',
 '14ce0c87-a3c7-42f9-8aa8-41c5d3bcdbb4',
 'ac3a8efe-d125-428f-b8e3-111c4ede75d4',
 'cd6f7621-7a60-4a9f-8e9c-e7cab7da5748',
 '999afc10-02fa-4c13-a973-83c68e57d57a',
 'e74e2b77-d26f-448a-b408-d9b0415654f4',
 '47bd4ad6-e5df-4331-94fe-a49dd37d1127',
 '807bcbf2-e552-4292-a43d-4135f2921fad',
 'bccdb5be-4c55-49dd-84ff-9e0f9e7f5ee5',
 '2b56f3bc-ba7c-40e7-9002-0621b91aeb59',
 '5a203e32-a331-450a-91d9-3088ff2889ad',
 '1d77b4a1-e21b-4cca-bf61-3a9af6ec242f',
 'c465c2ca-f4c3-4a56-9059-4ce941f899d6',
 '0f09a00f-5da7-49d0-8daf-d3ce06f7aebe',
 'fa1970fc-5df7-4617-b35e-c5e3d0afe207',
 'd2b15f43-31c4-4be6-9d74-b073920a074e',
 '5fc9d1ec-35bc-47d2-869b-d4a47e9d3977',
 'e7ad5738-3492-4102-8381-5ab4d3e389c1',
 'a8436424-e93b-4694-9f5d-dfe926595046',
 '0665536c-8b14-4b2f-b830-ec474f4f5f37',
 '8bc46ef0-6ed5-4f04-8a26-69539f3aa1e8',
 'c6f35619-2f61-47a7-90d4-5f66e919d95e',
 '467f5e4a-b1c0-42af-b81e-733d9aebb0c7',
 '56624cbe-bff8-

Create the retrieval chain

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

retriever = vector_store.as_retriever()

# Set up system prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    
])

# Create the question-answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

Query the rag bot with a question based on the CSV data

In [9]:
answer= rag_chain.invoke({"input": "which company does sheryl Baxter work for?"})
answer['answer']

'According to the context, Sheryl Baxter works for Rasmussen Group.'